In [ ]:
import pandas as pd
import numpy as np
from scipy import sparse
import pickle
import joblib
from pathlib import Path
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')


In [ ]:
# CREATE DIRECTORIES FIRST
Path("feature_store").mkdir(exist_ok=True)
Path("models").mkdir(exist_ok=True)

print("✅ Directories ready")

# LOAD DATA - FORCE INT64
print("🔄 Loading RetailRocket...")
events = pd.read_csv('events.csv')
events['visitorid'] = events['visitorid'].astype('int64')
events['itemid'] = events['itemid'].astype('int64')
events['timestamp'] = pd.to_datetime(events['timestamp'], unit='ms')

item_props1 = pd.read_csv('item_properties_part1.csv')
item_props2 = pd.read_csv('item_properties_part2.csv')

print(f"Events: {events.shape}")

# EVENT WEIGHTS
event_weights = {90: 1.0, 23: 1.5, 21: 3.0}
events['weight'] = events['event'].map(event_weights)
events['event_type'] = events['event'].map({90: 'view', 23: 'addtocart', 21: 'transaction'})

In [ ]:
# TEMPORAL SPLIT
events = events.sort_values('timestamp')
n_events = len(events)
train = events.iloc[:int(0.8*n_events)].copy()
valid = events.iloc[int(0.8*n_events):int(0.9*n_events)].copy()
test = events.iloc[int(0.9*n_events):].copy()

# SESSION WINDOWS (30min)
def create_sessions(df):
    df = df.copy()
    df['session_offset'] = df.groupby('visitorid')['timestamp'].diff().dt.total_seconds().fillna(0) / 60
    df['session_id'] = (df['session_offset'] > 30).cumsum()
    df['session_id'] = df['visitorid'].astype(str) + '_' + df['session_id'].astype(str)
    return df

In [ ]:
train = create_sessions(train)
valid = create_sessions(valid)

In [ ]:
# FIXED SPARSE MATRIX - NO dtype errors
def build_sparse(df, alpha=15):
    df = df.copy()
    df['visitorid'] = df['visitorid'].astype('int64')
    df['itemid'] = df['itemid'].astype('int64')
    
    # Aggregate + confidence
    interactions = df.groupby(['visitorid', 'itemid'])['weight'].sum().reset_index()
    interactions['confidence'] = 1 + alpha * interactions['weight']
    
    # Safe categorical encoding
    user_map = {uid: i for i, uid in enumerate(interactions['visitorid'].unique())}
    item_map = {iid: i for i, iid in enumerate(interactions['itemid'].unique())}
    
    user_codes = interactions['visitorid'].map(user_map).astype(np.int32)
    item_codes = interactions['itemid'].map(item_map).astype(np.int32)
    data = interactions['confidence'].astype(np.float32)
    
    sparse_mat = sparse.csr_matrix((data, (user_codes, item_codes)),
                                 shape=(len(user_map), len(item_map)))
    return sparse_mat

In [ ]:
train_sparse = build_sparse(train)
valid_sparse = build_sparse(valid)

In [ ]:
print(f"✅ Sparse matrices: train={train_sparse.shape}, valid={valid_sparse.shape}")

# SAVE FEATURE STORE
with open('feature_store/train_sparse.pkl', 'wb') as f:
    pickle.dump({'sparse': train_sparse}, f)
with open('feature_store/valid_sparse.pkl', 'wb') as f:
    pickle.dump({'sparse': valid_sparse}, f)

train.to_parquet('feature_store/train_sessions.parquet')
valid.to_parquet('feature_store/valid_sessions.parquet')

print("🎉 STAGE 0 COMPLETE!")